# PixiJoy neural inpaint (M1) — Colab GPU

**Quality only for Release-.** Smoke is contract-only.

1. Runtime → Change runtime type → **GPU**
2. Set `TRAIN_MODE = "quality"` below (default)
3. Stage ≥32 real photos into `data/rgb` + empty `data/READY` (Drive or upload)
4. Runtime → **Run all**

Fresh Run all starts **from scratch**. Resume needs Drive ckpt + `--resume`.

When finished, download `dist/inpaint.tflite` only if verify **OK**, then attach to
[Release- ml-packs-v1](https://github.com/Saikumar-S0906/Release-/releases/tag/ml-packs-v1).


In [ ]:
!nvidia-smi || true
import tensorflow as tf
print("TF", tf.__version__, "GPUs", tf.config.list_physical_devices("GPU"))


In [ ]:
import os, urllib.request, zipfile
from pathlib import Path
from google.colab import files

ROOT = Path("/content/pixijoy_inpaint")
ROOT.mkdir(parents=True, exist_ok=True)
os.chdir(ROOT)

ZIP_URL = "https://raw.githubusercontent.com/Saikumar-S0906/Release-/main/inpaint_lab.zip"
ok = False
try:
    urllib.request.urlretrieve(ZIP_URL, "inpaint_lab.zip")
    with zipfile.ZipFile("inpaint_lab.zip") as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
    print("extracted lab from Release- main", ok)
except Exception as e:
    print("zip fetch failed:", e)

if not ok:
    print("Upload inpaint_lab.zip")
    uploaded = files.upload()
    name = next(iter(uploaded))
    with zipfile.ZipFile(name) as z:
        z.extractall(".")
    ok = Path("train.py").is_file()
assert ok, "Could not load inpaint_lab.zip"


In [ ]:
# quality = product (Release-). smoke = contract only — never pin.
TRAIN_MODE = "quality"
RESUME = False  # True only if you restored dist/*.ckpt.keras from Drive

from pathlib import Path
from google.colab import files

data = Path("data")
rgb = data / "rgb"
rgb.mkdir(parents=True, exist_ok=True)

if TRAIN_MODE == "quality":
    print("QUALITY: need ≥32 real photos in data/rgb + data/READY")
    print("Option A: upload a zip with rgb/ (+ optional masks/) then we write READY")
    print("Option B: mount Drive and copy photos into data/rgb yourself")
    if not (data / "READY").is_file() or len(list(rgb.glob("*"))) < 32:
        print("Upload data zip (folder rgb/ inside) OR individual images…")
        uploaded = files.upload()
        for name in uploaded:
            if name.endswith(".zip"):
                import zipfile
                with zipfile.ZipFile(name) as z:
                    z.extractall("data")
            else:
                (rgb / Path(name).name).write_bytes(uploaded[name])
        # flatten if zip had top-level rgb/
        nested = data / "rgb"
        if nested.is_dir():
            pass
        (data / "READY").write_text("ok\n", encoding="utf-8")
    n = len([p for p in rgb.iterdir() if p.suffix.lower() in {".png",".jpg",".jpeg",".webp"}])
    print("rgb count", n)
    assert n >= 32, f"need ≥32 real photos, found {n}"
    assert (data / "READY").is_file()
else:
    print("SMOKE mode — will NOT be Release- quality")


In [ ]:
cmd = "python -u train.py --quality" if TRAIN_MODE == "quality" else "python -u train.py --smoke"
if RESUME:
    cmd += " --resume"
print("RUN", cmd)
get_ipython().system(cmd)
get_ipython().system("python -u export_tflite.py")
vcmd = "python -u verify_contract.py --quality" if TRAIN_MODE == "quality" else "python -u verify_contract.py"
get_ipython().system(vcmd)
from pathlib import Path
import hashlib
p = Path("dist/inpaint.tflite")
print("READY", p, p.stat().st_size, hashlib.sha256(p.read_bytes()).hexdigest())


In [ ]:
from google.colab import files
files.download("dist/inpaint.tflite")
print("Upload to GitHub Release ml-packs-v1 ONLY if verify printed OK")
